# 🎓 SKRIPSI S1 INFORMATIKA — UNIVERSITAS AMIKOM YOGYAKARTA
## Klasifikasi Kategori Indeks Massa Tubuh Menggunakan Simulasi Image Processing dan Machine Learning Berbasis Data Antropometri
*Body Mass Index Category Classification Using Image Processing Simulation and Machine Learning Based on Anthropometric Data*

---
**Identitas Peneliti:**
- **Nama Mahasiswa:** Satrio Raja Nabil Rizky
- **NIM:** 23.11.5538
- **Program Studi:** S1 Informatika — Fakultas Ilmu Komputer
- **Dosen Pembimbing:** Mulia Sulistiyono, S.Kom., M.Kom.

---
### tahapan Alur Penelitian (Pipeline):
1. **Instalasi & Import Library**
2. **Akuisisi & Integrasi Dataset Antropometri (`n = 3.700` Sampel)**
3. **Exploratory Data Analysis (EDA)**
4. **Simulasi Image Processing (Siluet Tubuh Sintetis $128 \times 128$ px) & Ekstraksi Fitur Morfologi**
5. **Feature Engineering (16 Fitur Gabungan) & Preprocessing (`StandardScaler` + Stratified Split)**
6. **Pemodelan 6 Algoritma Machine Learning & Evaluasi Performa**
7. **Tracking Eksperimen Menggunakan MLflow (18 Run Eksperimen)**
8. **Penyimpanan Artifact Model Terbaik (`.pkl`) untuk Deployment Aplikasi Streamlit**


In [ ]:
# ============================================================
# 1. INSTALASI LIBRARY
# ============================================================
!pip install mlflow streamlit opencv-python-headless imbalanced-learn -q
print('✅ Seluruh library berhasil diinstal!')


In [ ]:
# ============================================================
# 2. IMPORT LIBRARY
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import math
import pickle
import warnings
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    f1_score, precision_score, recall_score
)

warnings.filterwarnings('ignore')
print(f'✅ Semua library berhasil diimport | MLflow version: {mlflow.__version__}')


---
## 3. Akuisisi & Integrasi Dataset (`n = 3.700` Data Individu)
Menggabungkan tiga sumber dataset antropometri publik:
1. **Dataset `bmi.csv` Kaggle** (500 data individu)
2. **Dataset UCI Obesity Levels (`ObesityDataSet_raw_and_data_sinthetic.csv`)** (2.111 data individu)
3. **Dataset Kurasi `bodyPerformance.csv`** (1.089 data terkurasi untuk memperkaya representasi kelas)


In [ ]:
# ============================================================
# 3. LOAD & INTEGRASI DATASET (3.700 SAMPEL)
# ============================================================
import os

def bmi_index(bmi):
    if bmi < 16.0: return 0
    elif bmi < 18.5: return 1
    elif bmi < 25.0: return 2
    elif bmi < 30.0: return 3
    elif bmi < 40.0: return 4
    else: return 5

label_map = {
    0: 'Extremely Weak', 1: 'Weak', 2: 'Normal',
    3: 'Overweight', 4: 'Obesity', 5: 'Extreme Obesity'
}

# Jika bmi.csv yang diunggah sudah versi gabungan (3.700 baris), langsung muat.
# Jika masih versi 500 baris dan file dataset tambahan tersedia, otomatis gabungkan.
df_raw = pd.read_csv('bmi.csv')
if len(df_raw) == 500 and os.path.exists('ObesityDataSet_raw_and_data_sinthetic.csv') and os.path.exists('bodyPerformance.csv'):
    df_obesity_raw = pd.read_csv('ObesityDataSet_raw_and_data_sinthetic.csv')
    df_body_raw = pd.read_csv('bodyPerformance.csv')
    
    df_ob = pd.DataFrame()
    df_ob['Gender'] = df_obesity_raw['Gender'].str.strip()
    df_ob['Height'] = (df_obesity_raw['Height'] * 100).round().astype(int)
    df_ob['Weight'] = df_obesity_raw['Weight'].round().astype(int)
    df_ob['Index']  = (df_ob['Weight'] / ((df_ob['Height'] / 100) ** 2)).apply(bmi_index)
    
    df_bd = pd.DataFrame()
    df_bd['Gender'] = df_body_raw['gender'].map({'M': 'Male', 'F': 'Female'})
    df_bd['Height'] = df_body_raw['height_cm'].round()
    df_bd['Weight'] = df_body_raw['weight_kg'].round()
    df_bd = df_bd.dropna()
    df_bd['Height'] = df_bd['Height'].astype(int)
    df_bd['Weight'] = df_bd['Weight'].astype(int)
    df_bd['Index']  = (df_bd['Weight'] / ((df_bd['Height'] / 100) ** 2)).apply(bmi_index)
    
    df_bd_curated = pd.concat([
        df_bd[df_bd['Index'] == 0],
        df_bd[df_bd['Index'] == 1],
        df_bd[df_bd['Index'] == 2].sample(300, random_state=42),
        df_bd[df_bd['Index'] == 3].sample(100, random_state=42),
        df_bd[df_bd['Index'] == 4],
        df_bd[df_bd['Index'] == 5]
    ], ignore_index=True)
    
    df = pd.concat([df_raw[['Gender', 'Height', 'Weight', 'Index']], df_ob, df_bd_curated], ignore_index=True)
    df.to_csv('bmi.csv', index=False)
else:
    df = df_raw[['Gender', 'Height', 'Weight', 'Index']].copy()

df['Label'] = df['Index'].map(label_map)
print(f'✅ Dataset berhasil dimuat! Total baris: {df.shape[0]}, Total kolom: {df.shape[1]}')
print('\n=== DISTRIBUSI KATEGORI IMT ===')
print(df['Label'].value_counts())
print('\n=== PREVIEW 10 DATA PERTAMA ===')
print(df.head(10))


In [ ]:
# ============================================================
# 4. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Distribusi Kategori IMT
ax = axes[0]
sns.countplot(x='Index', data=df, palette='Set2', ax=ax, order=sorted(df['Index'].unique()))
ax.set_xticklabels([label_map[i] for i in sorted(df['Index'].unique())], rotation=30, ha='right')
ax.set_title('Distribusi Kategori IMT (n = 3.700)')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2., p.get_height()), ha='center', va='bottom')

# 2. Scatter Plot Height vs Weight
scatter = axes[1].scatter(df['Height'], df['Weight'], c=df['Index'], cmap='RdYlGn_r', alpha=0.65, s=25)
plt.colorbar(scatter, ax=axes[1], label='Index IMT')
axes[1].set_title('Sebaran Tinggi vs Berat Badan')
axes[1].set_xlabel('Height (cm)')
axes[1].set_ylabel('Weight (kg)')

# 3. Distribusi Gender
df['Gender'].value_counts().plot(kind='bar', ax=axes[2], color=['steelblue','salmon'], edgecolor='white')
axes[2].set_title('Distribusi Jenis Kelamin (Gender)')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 5. SIMULASI IMAGE PROCESSING — EKSTRAKSI FITUR SILUET TUBUH
# ============================================================
def generate_body_silhouette(height_cm, weight_kg, gender, img_size=128):
    img = np.zeros((img_size, img_size), dtype=np.uint8)
    h_norm = np.clip((height_cm - 140) / (200 - 140), 0.0, 1.2)
    w_norm = np.clip((weight_kg - 35)  / (165 - 35), 0.0, 1.2)
    body_h = max(20, int(0.5 * img_size + h_norm * 0.35 * img_size))
    body_w = max(8, int(0.15 * img_size + w_norm * 0.3  * img_size))
    shoulder_w = max(10, int(body_w * (1.3 if gender == 'Male' else 1.1)))
    cx, cy = img_size // 2, img_size // 2
    cv2.ellipse(img, (cx, cy), (body_w, body_h // 2), 0, 0, 360, 255, -1)
    cv2.ellipse(img, (cx, cy - body_h // 2 + 10), (shoulder_w, 10), 0, 0, 360, 255, -1)
    head_r = max(8, int(body_w * 0.5))
    cv2.circle(img, (cx, cy - body_h // 2 - head_r + 5), head_r, 255, -1)
    return img

def extract_image_features(img):
    _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    pixel_area = np.sum(binary > 0)
    coords = np.argwhere(binary > 0)
    if len(coords) > 0:
        y0, x0 = coords.min(axis=0); y1, x1 = coords.max(axis=0)
        bbox_h = y1 - y0 + 1; bbox_w = x1 - x0 + 1
    else:
        bbox_h = bbox_w = 1
    aspect_ratio  = bbox_h / (bbox_w + 1e-5)
    body_density  = pixel_area / ((bbox_h * bbox_w) + 1e-5)
    contours, _   = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter     = cv2.arcLength(contours[0], True) if contours else 0
    compactness   = (4 * math.pi * pixel_area) / (perimeter ** 2 + 1e-5)
    return {
        'pixel_area': pixel_area,
        'aspect_ratio': round(aspect_ratio, 4),
        'body_density': round(body_density, 4),
        'contour_perimeter': round(perimeter, 4),
        'compactness': round(compactness, 4)
    }

# Visualisasi Contoh Siluet
sample_cases = [
    (160, 50, 'Female', 'Normal'),
    (175, 95, 'Male',   'Overweight'),
    (165, 135, 'Female','Extreme Obesity'),
]
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
for ax, (h, w, g, lbl) in zip(axes, sample_cases):
    img = generate_body_silhouette(h, w, g)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{lbl}\n{g}, H={h}cm, W={w}kg', fontsize=10)
    ax.axis('off')
plt.suptitle('Simulasi Citra Biner Siluet Tubuh (128x128 piksel)', fontsize=13)
plt.tight_layout()
plt.show()

print('Mengekstrak 5 fitur morfologi citra untuk seluruh data...')
df_img = pd.DataFrame([extract_image_features(generate_body_silhouette(r.Height, r.Weight, r.Gender)) for _, r in df.iterrows()])
print('✅ Ekstraksi selesai!')
print(df_img.describe().round(4))


In [ ]:
# ============================================================
# 6. FEATURE ENGINEERING & PREPROCESSING (16 FITUR)
# ============================================================
df['Height_m']            = df['Height'] / 100
df['BMI']                 = df['Weight'] / (df['Height_m'] ** 2)
df['BMI_squared']         = df['BMI'] ** 2
df['Weight_Height_ratio'] = df['Weight'] / df['Height']
df['Weight_squared']      = df['Weight'] ** 2
df['Height_squared']      = df['Height'] ** 2
df['Gender_num']          = (df['Gender'] == 'Male').astype(int)
df['Height_category']     = pd.cut(df['Height'], bins=[0,155,170,185,300], labels=[0,1,2,3]).astype(int)
df['Weight_category']     = pd.cut(df['Weight'], bins=[0,60,80,100,300], labels=[0,1,2,3]).astype(int)

feat_cols = [
    'Gender_num','Height','Weight','Height_m','BMI','BMI_squared',
    'Weight_Height_ratio','Weight_squared','Height_squared',
    'Height_category','Weight_category',
    'pixel_area','aspect_ratio','body_density','contour_perimeter','compactness'
]

X = pd.concat([df[feat_cols[:11]], df_img], axis=1)
y = df['Index']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feat_cols)

splits = {
    'Eksperimen_1_80:20': train_test_split(X_scaled, y, test_size=0.20, random_state=42, stratify=y),
    'Eksperimen_2_75:25': train_test_split(X_scaled, y, test_size=0.25, random_state=42, stratify=y),
    'Eksperimen_3_70:30': train_test_split(X_scaled, y, test_size=0.30, random_state=42, stratify=y),
}

for ratio, (Xtr, Xte, ytr, yte) in splits.items():
    print(f'Skenario {ratio} -> Data Latih: {Xtr.shape[0]} baris | Data Uji: {Xte.shape[0]} baris')


In [ ]:
# ============================================================
# 7. PEMODELAN 6 ALGORITMA KLASIFIKASI & EVALUASI
# ============================================================
model_configs = {
    'Random Forest':      RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':  GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42),
    'Decision Tree':      DecisionTreeClassifier(max_depth=10, random_state=42),
    'SVM (RBF)':          SVC(kernel='rbf', C=10, gamma='scale', random_state=42, probability=True),
    'Logistic Regression':LogisticRegression(max_iter=2000, random_state=42),
    'KNN':                KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
}

target_names = ['Ext.Weak','Weak','Normal','Overweight','Obesity','Ext.Obesity']
X_train, X_test, y_train, y_test = splits['Eksperimen_1_80:20']
hasil = {}

print('=== HASIL EVALUASI PADA SKENARIO SPLIT 80:20 ===')
for nama, model in model_configs.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    hasil[nama] = {'model': model, 'y_pred': y_pred, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    print(f'{nama:<22} | Accuracy: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}%')


In [ ]:
# ============================================================
# 8. VISUALISASI CONFUSION MATRIX KE-6 ALGORITMA
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
cmaps = ['Blues', 'Greens', 'Purples', 'Oranges', 'YlOrRd', 'RdPu']

for ax, (nama, data), cmap in zip(axes.flatten(), hasil.items(), cmaps):
    cm = confusion_matrix(y_test, data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=target_names, yticklabels=target_names)
    ax.set_title(f"{nama}\nAccuracy: {data['accuracy']*100:.2f}% | F1: {data['f1']*100:.2f}%", fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Perbandingan Confusion Matrix 6 Algoritma Machine Learning (Split 80:20)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 9. TRACKING 18 RUN EKSPERIMEN DENGAN MLFLOW
# ============================================================
mlflow.set_experiment('Skripsi_Klasifikasi_IMT_23_11_5538')
all_mlflow_results = []

for split_name, (Xtr, Xte, ytr, yte) in splits.items():
    for nama, model_obj in model_configs.items():
        run_name = f"{nama.replace(' ', '_')}_{split_name}"
        with mlflow.start_run(run_name=run_name):
            model_obj.fit(Xtr, ytr)
            yp = model_obj.predict(Xte)
            acc  = accuracy_score(yte, yp)
            prec = precision_score(yte, yp, average='weighted', zero_division=0)
            rec  = recall_score(yte, yp, average='weighted', zero_division=0)
            f1   = f1_score(yte, yp, average='weighted', zero_division=0)
            
            mlflow.log_param('model_name', nama)
            mlflow.log_param('split_ratio', split_name)
            mlflow.log_param('n_samples', len(X_scaled))
            mlflow.log_metrics({'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_score': f1})
            
            all_mlflow_results.append({
                'Skenario Split': split_name,
                'Algoritma': nama,
                'Accuracy (%)': round(acc * 100, 2),
                'Precision (%)': round(prec * 100, 2),
                'Recall (%)': round(rec * 100, 2),
                'F1-Score (%)': round(f1 * 100, 2)
            })

df_rekap_mlflow = pd.DataFrame(all_mlflow_results)
print('=== REKAPITULASI 18 RUN EKSPERIMEN MLFLOW ===')
print(df_rekap_mlflow.to_string(index=False))


In [ ]:
# ============================================================
# 10. SIMPAN ARTIFACTS MODEL TERBAIK UNTUK STREAMLIT CLOUD
# ============================================================
best_rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
Xtr_best, Xte_best, ytr_best, yte_best = splits['Eksperimen_1_80:20']
best_rf.fit(Xtr_best, ytr_best)

with open('model_bmi.pkl', 'wb') as f:
    pickle.dump(best_rf, f)
with open('scaler_bmi.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('✅ File model_bmi.pkl, scaler_bmi.pkl, dan bmi.csv siap diunggah ke GitHub Streamlit!')
